# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

SELECT
JOIN
WHERE
GROUP BY
COUNT
ORDER BY

DataFrame
join
filter
groupBy
aggregation
sort/order

.alias()
.join()
.select()
.filter()
.groupBy()
.count()
.orderBy()

In [8]:
# Smaller DataFrame Lookup for just Patent ID and State
patent_states = patents.select("PATENT", "POSTATE")

# Use separate aliases because the patent state lookup is joined twice
c = citations.alias("c") # Citations Table
citing_patent = patent_states.alias("citing_patent")   # citing patent
cited_patent = patent_states.alias("cited_patent")   # cited patent

In [9]:
# Add the state of both the citing and cited patents
joined = (
    c.join(
        citing_patent,
        col("c.CITING") == col("citing_patent.PATENT"),
        "left"
    ).join(
        cited_patent,
        col("c.CITED") == col("cited_patent.PATENT"),
        "left"
    ).select(
        col("c.CITING"),
        col("c.CITED"),
        col("citing_patent.POSTATE").alias("CITING_STATE"),
        col("cited_patent.POSTATE").alias("CITED_STATE")
    )
)
# Cache this result since inspect/resuse next
joined.cache()
joined.show(10)

+-------+-----+------------+-----------+
| CITING|CITED|CITING_STATE|CITED_STATE|
+-------+-----+------------+-----------+
|4192521| 2366|        NULL|       NULL|
|4305315| 2366|          MN|       NULL|
|4253355| 2366|          MN|       NULL|
|5580635| 5156|          WI|       NULL|
|4976561| 5518|        NULL|       NULL|
|4480374| 5803|          MN|       NULL|
|5123817| 6620|        NULL|       NULL|
|4115020| 7240|        NULL|       NULL|
|4727698| 7253|          CA|       NULL|
|4108250| 7340|          IL|       NULL|
+-------+-----+------------+-----------+
only showing top 10 rows



In [10]:
# Keep only citations where both states exist and match.
same_state = joined.filter(
    (col("CITING_STATE").isNotNull()) &
    (col("CITED_STATE").isNotNull()) &
    (col("CITING_STATE") == col("CITED_STATE"))
)

# Count matching citations for each citing patent.
same_state_counts = (
    same_state
    .groupBy("CITING")
    .count()
    .withColumnRenamed("count", "SAME_STATE_COUNT")
    .cache()
)

# Check counts

same_state_counts.orderBy(
    col("SAME_STATE_COUNT").desc(),
    col("CITING").asc()
).show(10)

+-------+----------------+
| CITING|SAME_STATE_COUNT|
+-------+----------------+
|5959466|             125|
|5983822|             103|
|6008204|             100|
|5952345|              98|
|5958954|              96|
|5998655|              96|
|5936426|              94|
|5739256|              90|
|5913855|              90|
|5925042|              90|
+-------+----------------+
only showing top 10 rows



In [11]:
# Add the state count back to every original patent.
# Patents with no matching citations (NULL) receive a count of 0.
result = (
    patents
    .join(
        same_state_counts,
        patents.PATENT == same_state_counts.CITING,
        "left"
    )
    .drop("CITING")
    .fillna({"SAME_STATE_COUNT": 0})
    .orderBy(col("SAME_STATE_COUNT").desc())
)

result.show(10)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE_COUNT|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  NULL|   326|  4|    46|  159|       0|     1.0|   NULL|  0.6186|    NULL|  4.8868|  0.0455|   0.044|    NULL|    NULL|             125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  NULL|   114|  5|    55|  200|       0|   0.995|   NULL|  0.7201|    NULL|   12.45|     0.0|     0.0|    NULL|    NULL|             103|
